# Lab 1：静态页面与 API 数据采集

SmartPark DataLab；真实请求、分页采集与结果验证。

In [1]:
import json
from collections import Counter
from pathlib import Path
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_URL = "https://115.159.159.82"
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=Retry(total=3, backoff_factor=0.5, status_forcelist=[429, 500, 502, 503, 504])))


## 1. 判断加载方式与发现 Device API

In [2]:
api_url = urljoin(BASE_URL, "/api/devices")
first_response = session.get(api_url, params={"page": 1, "page_size": 50}, timeout=15)
first_response.raise_for_status()
data = first_response.json()
device_name = data["items"][0]["device_name"]

html_response = session.get(urljoin(BASE_URL, "/devices"), timeout=15)
html_response.raise_for_status()
print("Device page source contains first device name:", device_name in html_response.text)
print("Request URL:", first_response.url)
print("Request Method:", first_response.request.method)
print("Content-Type:", first_response.headers.get("Content-Type"))
print("JSON keys:", list(data.keys()))
print("First device:", data["items"][0])
assert device_name not in html_response.text


Device page source contains first device name: False
Request URL: https://115.159.159.82/api/devices?page=1&page_size=50
Request Method: GET
Content-Type: application/json
JSON keys: ['page', 'page_size', 'total', 'items']
First device: {'device_id': 'D00000001', 'company_id': 'C000001', 'device_name': '循环泵-000001-001', 'device_type': 'pump', 'model': 'SP-PUM-800', 'zone': '智造一区', 'install_date': '2021-09-19', 'rated_power_kw': 71.2, 'sampling_interval_s': 30, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-01T03:58:00'}


## 2. API 分页采集全部设备

In [3]:
total = data["total"]
page_size = data["page_size"]
total_pages = (total + page_size - 1) // page_size
print("total:", total, "page_size:", page_size, "total_pages:", total_pages)

devices = []
for page in range(1, total_pages + 1):
    if page == 1:
        page_data = data
    else:
        response = session.get(api_url, params={"page": page, "page_size": page_size}, timeout=15)
        response.raise_for_status()
        page_data = response.json()
    assert page_data["page"] == page, f"Page number mismatch at {page}"
    assert page_data["total"] == total, f"Total changed at {page}"
    devices.extend(page_data["items"])
    if page % 20 == 0 or page == total_pages:
        print(f"completed page {page}/{total_pages}; devices: {len(devices)}")

assert len(devices) == total


total: 5000 page_size: 50 total_pages: 100
completed page 20/100; devices: 1000
completed page 40/100; devices: 2000
completed page 60/100; devices: 3000
completed page 80/100; devices: 4000
completed page 100/100; devices: 5000


## 3. 从静态 HTML 提取企业，并沿“下一页”采集三页

In [4]:
def parse_company(card):
    details = {
        item.select_one("dt").get_text(strip=True): item.select_one("dd").get_text(strip=True)
        for item in card.select("dl > div")
    }
    return {
        "company_id": card["data-company-id"],
        "name": card.select_one("h2 a").get_text(strip=True),
        "industry": details["所属行业"],
        "zone": details["所在区域"],
        "device_count": int(details["设备数量"].split()[0]),
        "status": card.select_one(".badge").get_text(strip=True),
    }

companies = []
page_counts = []
current_url = urljoin(BASE_URL, "/companies")
first_next_href = None
for page in range(1, 4):
    response = session.get(current_url, timeout=15)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    cards = soup.select("article.company-card")
    page_counts.append(len(cards))
    companies.extend(parse_company(card) for card in cards)
    print(f"Company page {page}: {len(cards)} records; URL: {response.url}")
    assert len(cards) == 20
    if page < 3:
        next_link = soup.select_one("nav.pagination a:last-child")
        assert next_link is not None and next_link.get_text(" ", strip=True) == "下一页"
        next_href = next_link.get("href")
        if page == 1:
            first_next_href = next_href
        current_url = urljoin(current_url, next_href)

print("First company:", companies[0])
print("First next href:", first_next_href)


Company page 1: 20 records; URL: https://115.159.159.82/companies
Company page 2: 20 records; URL: https://115.159.159.82/companies?page=2
Company page 3: 20 records; URL: https://115.159.159.82/companies?page=3
First company: {'company_id': 'C000001', 'name': '临港精工001有限公司', 'industry': '汽车零部件', 'zone': '智造一区', 'device_count': 6, 'status': '运营中'}
First next href: /companies?page=2


## 4. 数量、唯一性及跨数据源一致性

In [5]:
device_counts = Counter(x["company_id"] for x in devices)
mismatches = [
    (c["company_id"], c["device_count"], device_counts[c["company_id"]])
    for c in companies
    if c["device_count"] != device_counts[c["company_id"]]
]
print("Company 总数:", len(companies))
print("唯一 company_id 数:", len({x["company_id"] for x in companies}))
print("Device 总数:", len(devices))
print("唯一 device_id 数:", len({x["device_id"] for x in devices}))
print("Company–Device 不一致企业数:", len(mismatches))
assert len(companies) == len({x["company_id"] for x in companies}) == 60
assert len(devices) == len({x["device_id"] for x in devices}) == 5000
assert not mismatches, mismatches[:5]


Company 总数: 60
唯一 company_id 数: 60
Device 总数: 5000
唯一 device_id 数: 5000
Company–Device 不一致企业数: 0


## 5. 保存 JSON

In [6]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
for filename, records in (("companies.json", companies), ("devices.json", devices)):
    target = data_dir / filename
    with target.open("w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(target, "records:", len(records), "bytes:", target.stat().st_size)


data\companies.json records: 60 bytes: 12001
data\devices.json records: 5000 bytes: 1977612
